In [16]:
from tensorflow import keras

# Define Inputs
despegados = keras.Input(shape=(None, 1), name="despegues")
cola = keras.Input(shape=(None, 1), name="en_cola")
plane = keras.Input(shape=(1,), name="plane")

# Process sequential inputs with SimpleRNN
despegados_out = keras.layers.SimpleRNN(8, return_sequences=False)(despegados)
cola_out = keras.layers.SimpleRNN(8, return_sequences=False)(cola)

# Process plane input with Dense layer
plane_out = keras.layers.Dense(8, activation="relu")(plane)

# Combine all processed inputs
combined = keras.layers.Concatenate()([despegados_out, cola_out, plane_out])

# Further processing
combined = keras.layers.Dense(8, activation="relu")(combined)
result = keras.layers.Dense(1)(combined)

# Define model
model = keras.Model(inputs=[despegados, cola, plane], outputs=[result])

# Print model summary
model.summary()

Model: "functional_6"

┏━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━┓
┃ Layer (type)        ┃ Output Shape      ┃    Param # ┃ Connected to      ┃
┡━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━┩
│ despegues           │ (None, None, 1)   │          0 │ -                 │
│ (InputLayer)        │                   │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ en_cola             │ (None, None, 1)   │          0 │ -                 │
│ (InputLayer)        │                   │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ plane (InputLayer)  │ (None, 1)         │          0 │ -                 │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ simple_rnn_23       │ (None, 8)         │         80 │ despegues[0][0]   │
│ (SimpleRNN)         │                   │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ simple_rnn_24       │ (None, 8)         │         80 │ en_cola[0][0]     │
│ (SimpleRNN)         │                   │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ dense_24 (Dense)    │ (None, 8)         │         16 │ plane[0][0]       │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ concatenate_14      │ (None, 24)        │          0 │ simple_rnn_23[0]… │
│ (Concatenate)       │                   │            │ simple_rnn_24[0]… │
│                     │                   │            │ dense_24[0][0]    │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ dense_25 (Dense)    │ (None, 8)         │        200 │ concatenate_14[0… │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ dense_26 (Dense)    │ (None, 1)         │          9 │ dense_25[0][0]    │
└─────────────────────┴───────────────────┴────────────┴───────────────────┘

 Total params: 385 (1.50 KB)

 Trainable params: 385 (1.50 KB)

 Non-trainable params: 0 (0.00 B)

In [34]:
import numpy as np
despegues = np.array([[10_00, 10_05, 10_10], [11_00, 11_07, 11_08], [12_00, 12_10, 12_15]])
colas = np.array([[10_11, 10_11, 10_12], [11_05, 11_06, 11_09], [12_14, 12_15, 12_17]])

# cuando se paran
planes = np.array([10_13, 11_10, 12_20])

# cuando despegan
y = np.array([10_20, 11_24, 12_25])

model.compile(optimizer=keras.optimizers.RMSprop(learning_rate=1e-2), loss=["mean_squared_error"])
model.fit([despegues, colas, planes], y, epochs=10)

Epoch 1/10
1/1 ━━━━━━━━━━━━━━━━━━━━ 2s 2s/step - loss: 686.1656
Epoch 2/10
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 56ms/step - loss: 15144.3330
Epoch 3/10
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 55ms/step - loss: 511.9965
Epoch 4/10
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 57ms/step - loss: 87.7909
Epoch 5/10
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 51ms/step - loss: 27.4936
Epoch 6/10
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 57ms/step - loss: 17.9437
Epoch 7/10
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 55ms/step - loss: 15.7391
Epoch 8/10
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 56ms/step - loss: 15.1144
Epoch 9/10
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 56ms/step - loss: 14.8964
Epoch 10/10
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 60ms/step - loss: 14.8174


In [35]:
model.predict([despegues, colas, planes])

1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 246ms/step


array([[1024.2126],
       [1119.2626],
       [1227.0511]], dtype=float32)

In [36]:
y

array([1020, 1124, 1225])